# Advanced Data Analytics — Stream Processing (Spark)  

This notebook supports module 5 on **Mining Data Streams**, including **windowing**, **stateful aggregation**, and **approximate distinct counting** using Spark Structured Streaming.  
You will observe how Spark processes unbounded data in real time and manages continuous updates across tumbling and sliding windows.

**Dataset**  
You will use the provided CSV files in `sample_data`.  
Each file represents a short burst of IoT sensor readings with columns:  
`timestamp, sensor_id, location, temperature, humidity`  
Example timestamp format: `YYYY-MM-DD HH:MM:SS`.

**How to Run the Experiments**  
1. Start the stream in the notebook.  
2. While it’s running, copy one CSV at a time from `sample_data` into the input folder to simulate live data.  
3. Observe how each new file updates the streaming output in the console.  
4. Experiment with parameters such as **window size**, **slide duration**, and **watermark** to see their effects on the results.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, window, avg, min as spark_min, max as spark_max,
    count, approx_count_distinct
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Tha paths for the streaming
INPUT_DIR = "streaming_input"        # You will drop CSVs here during the run
CHECKPOINT_BASE = "streaming_checkpoint"

# ensure directories exist
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_BASE, exist_ok=True)

# Streaming schema (no need to infer schema)
SENSOR_SCHEMA = StructType([
    StructField("timestamp",  StringType(), True),   # e.g., "2025-10-25 12:34:56"
    StructField("sensor_id",  StringType(), True),
    StructField("location",   StringType(), True),
    StructField("temperature",DoubleType(), True),
    StructField("humidity",   DoubleType(), True),
])

# Some default parameters (you can change these and experiment)
WINDOW_DURATION = "30 seconds"
SLIDE_DURATION  = "10 seconds"
WATERMARK       = "20 seconds"
TRIGGER_EVERY   = "5 seconds"

# run spark session
spark = (
    SparkSession.builder
    .appName("ADA-Streaming-Activity")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.port", "0")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.streaming.schemaInference", "false")
    .config("spark.hadoop.fs.defaultFS", "file:///")   # force local file system (otherwise, Spark by default tries to use HDFS)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark ready.")

In [ ]:
# if things go wrong and you want to stop the spark session, uncomment the following line and use it.
# spark.stop()

## 1) Create the streaming DataFrame (CSV directory source)

**What this does**  
- Reads CSV files from the input dir specified above as a *stream* (new files = new data).  
- Parses `timestamp` into `event_time` for event-time windowing.  
- Use `maxFilesPerTrigger=1` to simulate arrival pacing.  
- This models Spark’s “unbounded table” view of streams.


In [ ]:
# Base streaming DataFrame
# Read the code carefully and understand what it does.
base_stream = (
    spark.readStream
         .schema(SENSOR_SCHEMA)
         .option("header", "true")
         .option("maxFilesPerTrigger", 1)   # to simulate gradual arrival
         .csv(INPUT_DIR)
         .withColumn("event_time", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
)

# Peek at the schema (static printout)
base_stream.printSchema()

## 2) Tumbling Windows over Time (Non-Overlapping)

**Goal**  
Observe how Spark groups streaming records into **fixed time windows** and computes average temperature, average humidity, and count of rows within each window.  
This helps visualize how **tumbling windows** divide event-time data into non-overlapping intervals.

**Answer the following questions:**  
- How do results change when you set `WINDOW_DURATION` to **30s** vs **60s**?  
- What does each “Batch” in the console output represent?  
- Looking at the number of rows that you will compute below what do you notice across batches?  
- Notice the `outputMode='update'`? What would happen if we used `'append'` instead? **If you want to test it, stop q1 and remove the checkpoints and see what happens.**

In [ ]:
# TODO: FILL IN THE BLANKS WITH THE CORRECT CODE
# Compute the average of temperature and humidity in each window and name them to avg_temperature and avg_humidity 
# and count the number of rows in each window and name it rows
exp1 = (
    base_stream
      .groupBy(window(col("event_time"), WINDOW_DURATION))
      .agg(
          # TODO: Write your code here
      )
      .select(
          col("window.start").alias("window_start"),
          col("window.end").alias("window_end"),
          "avg_temperature", "avg_humidity", "rows"
      )
)


q1 = (
    exp1.writeStream
        .format("console")
        .outputMode("update")                      # in-progress updates are visible
        .option("truncate", "false")
        .option("numRows", "40")
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/exp1_tumbling")
        .trigger(processingTime=TRIGGER_EVERY)
        .start()
)

print("Exp1 started.\n Add CSVs to INPUT_DIR.\n Run q1.stop() to stop.\n")


### If you want to stop and test different setups
You can stop the query and remove the checkpoints it created using the following code (uncomment it)

In [ ]:
# q1.stop()
# !rm -rf streaming_checkpoint/exp1_tumbling

In [ ]:
# optioanl: Remove the input files
# !rm -rf streaming_input/*

### Remove the input files
For the next experiemt, remove the input files to make the input folder (source of streaming) empty.  
You can run the following to do that.

In [ ]:
# !rm -rf streaming_input/*

## 3) Sliding Windows per Sensor with Watermarking

**What to Do**  
Run the code block for `exp2`, then start copying the CSV files into the input folder one at a time.  
Observe the console output as new batches appear.  
Afterward, try copying one file that contains older timestamps to see how Spark handles late data.

**Answer the following questions:**  
- How do results differ from the tumbling windows in Experiment 1?  
- Why do some readings appear in multiple windows?  
- What happens if you copy a CSV file with older timestamps after the stream has advanced?  
- Why does skipping one file (e.g., adding 001, 002, 004, then 003 later) produce no update?  

**Note:** The following cell of parameters are deliberately copied here so that you can easily access.

In [ ]:
# Some default parameters (you can change these and experiment)
WINDOW_DURATION = "30 seconds"
SLIDE_DURATION  = "10 seconds"
WATERMARK       = "20 seconds"
TRIGGER_EVERY   = "5 seconds"

In [ ]:
# TODO: FILL IN THE BLANKS WITH THE CORRECT CODE
# Compute the average of temperature, the minimum and maximum temperature, and the number of readings in each window and sensor_id and 
# name them avg_temp, min_temp, max_temp, and readings respectively.
exp2 = (
    base_stream
      .withWatermark("event_time", WATERMARK)
      .groupBy(window(col("event_time"), WINDOW_DURATION, SLIDE_DURATION), col("sensor_id"))
      .agg(
          # TODO: Write your code here
      )
      .select(
          col("window.start").alias("window_start"),
          col("window.end").alias("window_end"),
          "sensor_id", "avg_temp", "min_temp", "max_temp", "readings"
      )
)

q2 = (
    exp2.writeStream
        .format("console")
        .outputMode("update")
        .option("truncate", "false")
        .option("numRows", "40")
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/exp2_sliding")
        .trigger(processingTime=TRIGGER_EVERY)
        .start()
)

print("Exp2 started.\n Add CSVs to INPUT_DIR.\n Run q1.stop() to stop.\n")

In [ ]:
# For stopping and testing different setups
# q2.stop()
# !rm -rf streaming_checkpoint/exp2_sliding

## 4) Approximate Distinct Counting per Location [OPTIONAL]

This experiment demonstrates **probabilistic distinct counting** using Spark’s `approx_count_distinct()` function.  
It estimates how many unique sensors have reported readings within each 60-second window, grouped by location.

Spark uses a **HyperLogLog** algorithm for this operation, which provides an approximate count with controlled error margins.  
The parameter in `approx_count_distinct(sensor_id, rsd)` defines the **relative standard deviation (RSD)**, a lower RSD means higher accuracy but greater memory cost.

Two estimations are shown for comparison:  
- `approx_count_distinct(sensor_id, 0.05)` → more accurate, more state retained  
- `approx_count_distinct(sensor_id, 0.20)` → less accurate, smaller state footprint

This technique allows Spark to perform scalable, memory-efficient distinct counting on large or continuous streams,  
trading exactness for performance—an important concept in real-time analytics.

In [ ]:
# 4) Approx distinct counting by location (single 60s window)
DISTINCT_WINDOW = "60 seconds"

exp3 = (
    base_stream
      .withWatermark("event_time", WATERMARK)
      .groupBy(window(col("event_time"), DISTINCT_WINDOW), col("location"))
      .agg(
          count("*").alias("total_events"),
          approx_count_distinct("sensor_id", 0.05).alias("approx_distinct_rsd_005"),
          approx_count_distinct("sensor_id", 0.20).alias("approx_distinct_rsd_020")
      )
      .select(
          col("window.start").alias("window_start"),
          col("window.end").alias("window_end"),
          "location", "total_events", "approx_distinct_rsd_005", "approx_distinct_rsd_020"
      )
)

q3 = (
    exp3.writeStream
        .format("console")
        .outputMode("update")
        .option("truncate", "false")
        .option("numRows", "40")
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/exp3_approx_distinct")
        .trigger(processingTime=TRIGGER_EVERY)
        .start()
)

print("Exp3 started. Add CSVs to INPUT_DIR. Run q3.stop() to stop.")

## Some useful code
If you want to stop all the streaming queries at once:

In [ ]:
for q in spark.streams.active:
    q.stop()

In [ ]:
# Do this to stop the spark session
spark.stop()